# CA2 - Variational Autoencoder Implementation

This notebook implements the VAE and β-VAE models for the dSprites dataset as required in Question 2 of CA2.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.decomposition import PCA
from scipy.stats import multivariate_normal
import requests
from tqdm import tqdm

# Set random seeds for reproducibility
def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## dSprites Dataset Loading and Preprocessing

In [ ]:
class DSpritesDataset(Dataset):
    def __init__(self, transform=None):
        # Download dSprites dataset
        url = "https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz"
        data_path = "dsprites.npz"
        
        if not os.path.exists(data_path):
            print("Downloading dSprites dataset...")
            response = requests.get(url)
            with open(data_path, 'wb') as f:
                f.write(response.content)
        
        data = np.load(data_path)
        self.images = data['imgs'].astype(np.float32)
        self.latent_values = data['latents_values']
        self.latent_classes = data['latents_classes']
        self.metadata = data['metadata'][()]
        
        self.transform = transform
        
        # Reshape images to add channel dimension
        if len(self.images.shape) == 3:
            self.images = self.images.reshape(-1, 1, 64, 64)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        latent = self.latent_values[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, latent

# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
])

full_dataset = DSpritesDataset(transform=transform)
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size], 
    generator=torch.Generator().manual_seed(42)
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Dataset sizes - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Show some sample images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(8):
    img, _ = full_dataset[i]
    axes[i//4, i%4].imshow(img.squeeze(), cmap='gray')
    axes[i//4, i%4].axis('off')
plt.tight_layout()
plt.savefig('../pictures/dsprites_samples.png', dpi=300, bbox_inches='tight')
plt.show()

## VAE Model Architecture

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=10):
        super(VAE, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 64x64 -> 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 32x32 -> 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 16x16 -> 8x8
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),  # 8x8 -> 4x4
            nn.ReLU(),
            nn.Flatten()
        )
        
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        # Decoder
        self.decoder_input = nn.Linear(latent_dim, 256 * 4 * 4)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),    # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),     # 32x32 -> 64x64
            nn.Sigmoid()
        )
        
    def encode(self, x):
        x = self.encoder(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        x = self.decoder_input(z)
        x = x.view(-1, 256, 4, 4)
        x = self.decoder(x)
        return x
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        return reconstructed, mu, logvar

# Loss functions
def vae_loss(reconstructed, original, mu, logvar, beta=1.0):
    # Reconstruction loss (BCE)
    recon_loss = nn.functional.binary_cross_entropy(
        reconstructed, original, reduction='sum'
    )
    
    # KL divergence
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

## Training Functions

In [ ]:
def train_vae(model, train_loader, val_loader, beta=1.0, num_epochs=50, lr=1e-4):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    train_recon_losses = []
    train_kl_losses = []
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_kl = 0
        
        for batch_x, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            batch_x = batch_x.to(device)
            
            optimizer.zero_grad()
            reconstructed, mu, logvar = model(batch_x)
            
            loss, recon_loss, kl_loss = vae_loss(reconstructed, batch_x, mu, logvar, beta)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device)
                reconstructed, mu, logvar = model(batch_x)
                loss, _, _ = vae_loss(reconstructed, batch_x, mu, logvar, beta)
                val_loss += loss.item()
        
        train_losses.append(epoch_loss / len(train_loader))
        val_losses.append(val_loss / len(val_loader))
        train_recon_losses.append(epoch_recon / len(train_loader))
        train_kl_losses.append(epoch_kl / len(train_loader))
        
        print(f"Epoch {epoch+1}: Train Loss = {train_losses[-1]:.2f}, Val Loss = {val_losses[-1]:.2f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'../models/best_vae_beta_{beta}.pth')
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_recon': train_recon_losses,
        'train_kl': train_kl_losses
    }

# Create models directory
os.makedirs('../models', exist_ok=True)

## Train Standard VAE (β=1)

In [ ]:
print("Training standard VAE (β=1)...")
vae_model = VAE(latent_dim=10)
results_beta1 = train_vae(vae_model, train_loader, val_loader, beta=1.0, num_epochs=50)

## Train β-VAE models (β=2, β=4)

In [ ]:
print("Training β-VAE (β=2)...")
vae_beta2 = VAE(latent_dim=10)
results_beta2 = train_vae(vae_beta2, train_loader, val_loader, beta=2.0, num_epochs=50)

print("Training β-VAE (β=4)...")
vae_beta4 = VAE(latent_dim=10)
results_beta4 = train_vae(vae_beta4, train_loader, val_loader, beta=4.0, num_epochs=50)

## Generate Training Curves

In [ ]:
# Plot training curves for all models
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Reconstruction Loss
axes[0, 0].plot(results_beta1['train_recon'], label='β=1', color='blue')
axes[0, 0].plot(results_beta2['train_recon'], label='β=2', color='red')
axes[0, 0].plot(results_beta4['train_recon'], label='β=4', color='green')
axes[0, 0].set_title('Reconstruction Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# KL Divergence
axes[0, 1].plot(results_beta1['train_kl'], label='β=1', color='blue')
axes[0, 1].plot(results_beta2['train_kl'], label='β=2', color='red')
axes[0, 1].plot(results_beta4['train_kl'], label='β=4', color='green')
axes[0, 1].set_title('KL Divergence')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Total Loss
axes[0, 2].plot(results_beta1['train_losses'], label='β=1', color='blue')
axes[0, 2].plot(results_beta2['train_losses'], label='β=2', color='red')
axes[0, 2].plot(results_beta4['train_losses'], label='β=4', color='green')
axes[0, 2].set_title('Total Loss')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Validation losses
axes[1, 0].plot(results_beta1['val_losses'], label='β=1', color='blue')
axes[1, 0].plot(results_beta2['val_losses'], label='β=2', color='red')
axes[1, 0].plot(results_beta4['val_losses'], label='β=4', color='green')
axes[1, 0].set_title('Validation Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Final losses comparison
models = ['β=1', 'β=2', 'β=4']
final_recon = [results_beta1['train_recon'][-1], results_beta2['train_recon'][-1], results_beta4['train_recon'][-1]]
final_kl = [results_beta1['train_kl'][-1], results_beta2['train_kl'][-1], results_beta4['train_kl'][-1]]
final_total = [results_beta1['train_losses'][-1], results_beta2['train_losses'][-1], results_beta4['train_losses'][-1]]

x = np.arange(len(models))
width = 0.25

axes[1, 1].bar(x - width, final_recon, width, label='Reconstruction', alpha=0.7)
axes[1, 1].bar(x, final_kl, width, label='KL', alpha=0.7)
axes[1, 1].bar(x + width, final_total, width, label='Total', alpha=0.7)
axes[1, 1].set_title('Final Losses Comparison')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(models)
axes[1, 1].legend()

# Hide the last subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('../pictures/training_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Generate Reconstructions

In [ ]:
def generate_reconstructions(model, test_loader, num_samples=8):
    model.eval()
    
    # Get some test samples
    test_iter = iter(test_loader)
    test_batch, _ = next(test_iter)
    test_samples = test_batch[:num_samples].to(device)
    
    with torch.no_grad():
        reconstructed, _, _ = model(test_samples)
    
    # Plot original vs reconstructed
    fig, axes = plt.subplots(2, num_samples, figsize=(16, 4))
    
    for i in range(num_samples):
        # Original
        axes[0, i].imshow(test_samples[i].cpu().squeeze(), cmap='gray')
        axes[0, i].set_title('Original')
        axes[0, i].axis('off')
        
        # Reconstructed
        axes[1, i].imshow(reconstructed[i].cpu().squeeze(), cmap='gray')
        axes[1, i].set_title('Reconstructed')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    return fig

# Generate reconstructions for all models
models_dict = {
    'β=1': vae_model,
    'β=2': vae_beta2,
    'β=4': vae_beta4
}

for name, model in models_dict.items():
    fig = generate_reconstructions(model, test_loader)
    plt.savefig(f'../pictures/reconstructions_{name.replace("=", "")}.png', dpi=300, bbox_inches='tight')
    plt.close()

## MIG (Mutual Information Gap) Calculation

In [ ]:
def compute_mig(model, data_loader, num_samples=10000):
    model.eval()
    latents = []
    ground_truths = []
    
    with torch.no_grad():
        count = 0
        for batch_x, batch_latent in data_loader:
            if count >= num_samples:
                break
            batch_x = batch_x.to(device)
            mu, _ = model.encode(batch_x)
            latents.append(mu.cpu().numpy())
            ground_truths.append(batch_latent.numpy())
            count += len(batch_x)
    
    latents = np.concatenate(latents)[:num_samples]
    ground_truths = np.concatenate(ground_truths)[:num_samples]
    
    # dSprites factors: shape, scale, orientation, x_pos, y_pos
    factor_names = ['shape', 'scale', 'orientation', 'x_pos', 'y_pos']
    
    mig_scores = {}
    
    for factor_idx, factor_name in enumerate(factor_names):
        factor_values = ground_truths[:, factor_idx]
        
        # Compute mutual information for each latent dimension
        mi_scores = []
        for latent_idx in range(latents.shape[1]):
            latent_dim = latents[:, latent_idx]
            
            # Discretize for mutual information calculation
            latent_bins = np.linspace(latent_dim.min(), latent_dim.max(), 20)
            factor_bins = np.unique(factor_values)
            
            latent_discrete = np.digitize(latent_dim, latent_bins)
            
            # Compute mutual information using histogram
            joint_hist = np.histogram2d(latent_discrete, factor_values, 
                                       bins=[len(np.unique(latent_discrete)), len(factor_bins)])[0]
            joint_hist = joint_hist / joint_hist.sum()
            
            marginal_latent = joint_hist.sum(axis=1)
            marginal_factor = joint_hist.sum(axis=0)
            
            # Mutual information
            mi = 0
            for i in range(joint_hist.shape[0]):
                for j in range(joint_hist.shape[1]):
                    if joint_hist[i, j] > 0:
                        mi += joint_hist[i, j] * np.log(joint_hist[i, j] / (marginal_latent[i] * marginal_factor[j]))
            
            mi_scores.append(mi)
        
        # Compute MIG for this factor
        mi_scores = np.array(mi_scores)
        if len(mi_scores) > 1:
            sorted_mi = np.sort(mi_scores)
            mig = (sorted_mi[-1] - sorted_mi[-2]) / len(latents[0])
        else:
            mig = mi_scores[0] / len(latents[0])
        
        mig_scores[factor_name] = mig
    
    return mig_scores

# Compute MIG for all models
mig_results = {}
for name, model in models_dict.items():
    print(f"Computing MIG for {name}...")
    mig_results[name] = compute_mig(model, test_loader)
    print(f"{name} MIG scores: {mig_results[name]}")

# Save MIG results
np.save('../results/mig_results.npy', mig_results)

## PCA Analysis of Latent Space

In [ ]:
def plot_latent_pca(model, data_loader, model_name, num_samples=5000):
    model.eval()
    latents = []
    ground_truths = []
    
    with torch.no_grad():
        count = 0
        for batch_x, batch_latent in data_loader:
            if count >= num_samples:
                break
            batch_x = batch_x.to(device)
            mu, _ = model.encode(batch_x)
            latents.append(mu.cpu().numpy())
            ground_truths.append(batch_latent.numpy())
            count += len(batch_x)
    
    latents = np.concatenate(latents)[:num_samples]
    ground_truths = np.concatenate(ground_truths)[:num_samples]
    
    # PCA on latent space
    pca = PCA(n_components=2)
    latent_pca = pca.fit_transform(latents)
    
    # Plot PCA colored by each factor
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    factor_names = ['shape', 'scale', 'orientation', 'x_pos', 'y_pos']
    
    for i, factor_name in enumerate(factor_names):
        factor_values = ground_truths[:, i]
        
        ax = axes[i//3, i%3]
        scatter = ax.scatter(latent_pca[:, 0], latent_pca[:, 1], 
                           c=factor_values, cmap='viridis', alpha=0.6, s=1)
        ax.set_title(f'{model_name} - {factor_name}')
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        plt.colorbar(scatter, ax=ax)
    
    # Hide the last subplot if needed
    if len(factor_names) < 6:
        axes[-1, -1].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'../pictures/pca_{model_name.replace("=", "")}.png', dpi=300, bbox_inches='tight')
    plt.close()

# Generate PCA plots for all models
for name, model in models_dict.items():
    print(f"Generating PCA plot for {name}...")
    plot_latent_pca(model, test_loader, name)

print("VAE implementation completed!")
print("Generated files:")
print("- training_curves_comparison.png")
print("- reconstructions_β1.png, reconstructions_β2.png, reconstructions_β4.png")
print("- pca_β1.png, pca_β2.png, pca_β4.png")
print("- mig_results.npy")